In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import (
    train_test_split,
    RandomizedSearchCV
)

from sklearn.ensemble import RandomForestRegressor

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

In [2]:
# LOAD DATA

df = pd.read_csv("/content/used_device_data.csv")
print("Dataset Shape:", df.shape)

Dataset Shape: (3454, 15)


In [3]:
# FEATURE ENGINEERING
CURRENT_YEAR = 2026

df["device_age"] = CURRENT_YEAR - df["release_year"]

df["usage_ratio"] = (
    df["days_used"] /
    (df["device_age"] * 365 + 1)
)

df["camera_total"] = (
    df["rear_camera_mp"] +
    df["front_camera_mp"]
)

df["battery_per_weight"] = (
    df["battery"] /
    df["weight"]
)

df["ram_storage_ratio"] = (
    df["ram"] /
    df["internal_memory"]
)

In [4]:
# BINARY ENCODING
df["4g"] = df["4g"].map({
    "yes": 1,
    "no": 0
})

df["5g"] = df["5g"].map({
    "yes": 1,
    "no": 0
})

In [5]:
# Features & TARGET (LOG TARGET)
X = df.drop(
    columns=[
        "normalized_used_price"
    ]
)
y = df["normalized_used_price"]

In [6]:
# ONE HOT ENCODING
X = pd.get_dummies(
    X,
    columns=["device_brand", "os"],
    drop_first=True
)

In [7]:
# TRAIN TEST SPLIT
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

In [8]:
# RANDOM FOREST
rf = RandomForestRegressor(
    random_state=42,
    n_jobs=-1
)

In [9]:
# PARAMETER GRID
param_grid = {

    "n_estimators": [
        200,
        300,
        500
    ],

    "max_depth": [
        10,
        15,
        20,
        None
    ],

    "min_samples_split": [
        2,
        5,
        10
    ],

    "min_samples_leaf": [
        1,
        2,
        4
    ]
}


In [10]:
# HYPERPARAMETER SEARCH
search = RandomizedSearchCV(

    estimator=rf,

    param_distributions=param_grid,

    n_iter=10,

    cv=3,

    scoring="r2",

    random_state=42,

    n_jobs=-1,

    verbose=2
)

search.fit(X_train, y_train)

Fitting 3 folds for each of 10 candidates, totalling 30 fits


RandomizedSearchCV(cv=3,
                   estimator=RandomForestRegressor(n_jobs=-1, random_state=42),
                   n_jobs=-1,
                   param_distributions={'max_depth': [10, 15, 20, None],
                                        'min_samples_leaf': [1, 2, 4],
                                        'min_samples_split': [2, 5, 10],
                                        'n_estimators': [200, 300, 500]},
                   random_state=42, scoring='r2', verbose=2)

In [11]:
# BEST MODEL
best_rf = search.best_estimator_

print("\nBest Parameters:")
print(search.best_params_)


Best Parameters:
{'n_estimators': 300, 'min_samples_split': 5, 'min_samples_leaf': 1, 'max_depth': 10}


In [12]:
# PREDICTIONS
y_pred = best_rf.predict(X_test)

In [13]:
# EVALUATION
mae = mean_absolute_error(
    y_test,
    y_pred
)

rmse = np.sqrt(
    mean_squared_error(
        y_test,
        y_pred
    )
)

r2 = r2_score(
    y_test,
    y_pred
)

print("\n===== RESULTS =====")

print("MAE :", round(mae, 4))
print("RMSE:", round(rmse, 4))
print("R2 :", round(r2, 4))


===== RESULTS =====
MAE : 0.171
RMSE: 0.2135
R2 : 0.8595


In [14]:
# FEATURE IMPORTANCE
importance = pd.DataFrame({
    "Feature": X.columns,
    "Importance": best_rf.feature_importances_
})

importance = importance.sort_values(
    by="Importance",
    ascending=False
)

print("\n===== TOP 20 FEATURES =====")
print(importance.head(20))


===== TOP 20 FEATURES =====
                 Feature  Importance
11  normalized_new_price    0.588694
0            screen_size    0.139651
5        internal_memory    0.107209
14          camera_total    0.044229
7                battery    0.028830
8                 weight    0.025134
3         rear_camera_mp    0.015385
15    battery_per_weight    0.010454
13           usage_ratio    0.008101
4        front_camera_mp    0.007732
10             days_used    0.007392
16     ram_storage_ratio    0.004433
6                    ram    0.001875
12            device_age    0.001294
9           release_year    0.001255
40   device_brand_Others    0.000717
1                     4g    0.000697
43  device_brand_Samsung    0.000628
29  device_brand_Karbonn    0.000535
17  device_brand_Alcatel    0.000446


In [15]:
# SAMPLE PREDICTION
sample = X_test.iloc[[0]]

pred_log = best_rf.predict(sample)[0]

actual_log = y_test.iloc[0]

print("\nPredicted Log Price :", round(pred_log, 4))
print("Actual Log Price    :", round(actual_log, 4))


Predicted Log Price : 4.0399
Actual Log Price    : 3.9742


In [16]:
# Convert to actual price just for understanding

print("\nPredicted Price : $", round(np.exp(pred_log), 2))
print("Actual Price    : $", round(np.exp(actual_log), 2))


Predicted Price : $ 56.82
Actual Price    : $ 53.21
